# Resident Risk Predictive

## Problem Framing

**Business question:** Which residents are most likely to experience a near-term incident?

**Operational decision supported:** Trigger earlier staff intervention for residents with elevated short-term safety risk.

**Primary users:** case managers, operations staff

**Target summary:** Current predictive label: `label_incident_next_30d` from resident-monthly snapshots.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='resident_risk',
    dataset_name='resident_monthly_features',
    predictive_impl='resident_risk',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


,resident_id,snapshot_month,safehouse_id,case_status,case_category,sex,initial_risk_level,months_since_admission,age_years_at_snapshot,subcategory_flag_count,...,future_window_complete_60d,future_window_complete_90d,future_window_complete_120d,label_incident_next_30d,label_case_prioritization_next_60d,label_counseling_progress_next_90d,label_education_improvement_next_120d,label_wellbeing_deterioration_next_90d,label_supportive_home_visit_next_120d,label_reintegration_complete_next_90d
0,50,2023-01-01,4,Active,Neglected,F,High,0,11.90,2,...,True,True,True,0,False,False,True,False,False,False
1,45,2023-02-01,3,Transferred,Abandoned,F,Medium,0,14.99,2,...,True,True,True,0,False,False,True,False,True,False
2,23,2023-02-01,5,Closed,Foundling,F,High,0,9.07,1,...,True,True,True,0,True,True,True,False,False,False
3,13,2023-02-01,2,Closed,Surrendered,F,Medium,0,8.73,1,...,True,True,True,0,True,True,True,False,False,False
4,50,2023-02-01,4,Active,Neglected,F,High,1,11.98,2,...,True,True,True,0,False,False,True,False,False,False
5,23,2023-03-01,5,Closed,Foundling,F,High,1,9.15,1,...,True,True,True,0,True,True,True,False,False,False
6,29,2023-03-01,8,Transferred,Abandoned,F,Medium,0,12.08,1,...,True,True,True,0,True,True,True,False,False,False
7,2,2023-03-01,3,Closed,Surrendered,F,Medium,0,14.85,2,...,True,True,True,0,False,False,True,False,True,False
8,13,2023-03-01,2,Closed,Surrendered,F,Medium,1,8.80,1,...,True,True,True,0,True,True,True,False,False,False
9,36,2023-03-01,1,Active,Surrendered,F,High,0,12.06,2,...,True,True,True,0,True,False,True,False,False,False


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_incident_next_30d` from resident-monthly snapshots.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [3]:
evaluation = load_evaluation_bundle('resident_risk')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'logistic_regression',
 'train_rows': 756,
 'test_rows': 777,
 'target_col': 'label_incident_next_30d',
 'split_col': 'snapshot_month',
 'selection_metric': 'average_precision',
 'cutoff_date': '2025-04-01',
 'task_type': 'classification',
 'sample_count': 777.0,
 'positive_count': 36.0,
 'positive_rate': 0.04633204633204633,
 'accuracy': 0.8584298584298584,
 'balanced_accuracy': 0.5822087269455691,
 'precision': 0.10638297872340426,
 'recall': 0.2777777777777778,
 'f1': 0.15384615384615385,
 'roc_auc': 0.7580971659919028,
 'average_precision': 0.1231033816134102}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,resident_risk,logistic_regression,777.0,36.0,0.046332,0.886744,0.663124,0.182927,0.416667,0.254237,0.903959,0.218096,NaN,NaN,NaN
1,resident_risk,random_forest_classifier,777.0,36.0,0.046332,0.953668,0.500000,0.000000,0.000000,0.000000,0.855357,0.186471,NaN,NaN,NaN
2,resident_risk,dummy_classifier,777.0,36.0,0.046332,0.953668,0.500000,0.000000,0.000000,0.000000,0.500000,0.046332,NaN,NaN,NaN


In [5]:
summarize_frame(cv_summary)


,pipeline_name,model_name,fold_mean,fold_std,sample_count_mean,sample_count_std,positive_count_mean,positive_count_std,positive_rate_mean,positive_rate_std,...,roc_auc_std,average_precision_mean,average_precision_std,n_splits,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,resident_risk,logistic_regression,3.0,1.581139,151.2,0.447214,11.4,0.547723,0.075392,0.003489,...,0.069543,0.202432,0.103201,5,NaN,NaN,NaN,NaN,NaN,NaN
1,resident_risk,random_forest_classifier,3.0,1.581139,151.2,0.447214,11.4,0.547723,0.075392,0.003489,...,0.059066,0.178794,0.044773,5,NaN,NaN,NaN,NaN,NaN,NaN
2,resident_risk,dummy_classifier,3.0,1.581139,151.2,0.447214,11.4,0.547723,0.075392,0.003489,...,0.000000,0.075392,0.003489,5,NaN,NaN,NaN,NaN,NaN,NaN


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to case managers, operations staff?


## Deployment Notes

Recommended shared widgets:

* `risk_badge_widget`
* `ranked_table_widget`
* `explanation_chart_card`

Deployment checklist:

* Expose risk badges in resident detail views and triage dashboards.
* Use the same explanation card in watchlists and alert workflows.

Standard endpoint pattern:

* `POST /ml/predict/resident_risk`
* `POST /ml/score-batch/resident_risk`
